In [6]:
from torch.utils.data import Dataset
import torch
from collections import Counter
from nltk.util import ngrams
import torch.nn as nn

In [4]:
class BpeTokenizer:
  def __init__(self):
    self.merge_rules = []

  def train(self, text, vocab_size):
    vocab = list(set(newtext := list(text.replace(" ", "_").lower())))

    while len(vocab) < vocab_size:
      pary = ["".join(x) for x in ngrams(newtext, 2)]

      counter = Counter(pary)

      c = 0
      najcz = counter.most_common()[c][0]

      try:
        while najcz in vocab or "_" in najcz:
          c+= 1
          najcz = counter.most_common()[c][0]
      except:
        print("Nie mozna znalezc wiecej par. Przerywam")
        break

      self.merge_rules.append(najcz)
      #print(najcz)

      vocab.append(najcz)

      i = 0
      polaczone = []

      while i < len(newtext):
        if i < len(newtext)-1 and newtext[i] + newtext[i+1] == najcz:
          polaczone.append(najcz)
          i += 2
        else:
          polaczone.append(newtext[i])
          i += 1

      newtext = polaczone

      #print(newtext)
      #print(vocab)

  def tokenize(self, text):
    newtext = list(text.replace(" ", "_").lower())

    #print(self.merge_rules)


    if not self.merge_rules:
      raise ValueError("Model nie zostal wytrenowany lub masz dogshit training set!")

    for q in self.merge_rules:
      i = 0
      polaczone = []

      while i < len(newtext):
        if i < len(newtext)-1 and newtext[i] + newtext[i+1] == q:
          polaczone.append(q)
          i += 2
        else:
          polaczone.append(newtext[i])
          i += 1

      newtext = polaczone

    return newtext

tokenizer = BpeTokenizer()
tokenizer.train("Ala ma kota, a kot lubi pić mleko.", 20)

#print(tokenizer.tokenize("Ala ma kota, a kot lubi pić mleko."))


['ala', '_', 'ma', '_', 'kot', 'a', ',', '_', 'a', '_', 'kot', '_', 'l', 'u', 'b', 'i', '_', 'p', 'i', 'ć', '_', 'm', 'l', 'e', 'ko', '.']


In [5]:
class CustomDataset(Dataset):
  def __init__(self, text, tokenizer, vocab, context_size=2):
    super().__init__()
    self.samples = []

    tokens = tokenizer.tokenize(text)

    token_ids = [vocab[token] for token in tokens]

    for i in range(context_size, len(token_ids)-context_size):
      context = (token_ids[i-context_size:i] + token_ids[i+1:i+context_size+1])

      target = token_ids[i]

      self.samples.append((context, target))

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, idx):
    context, target = self.samples[idx]

    return (torch.tensor(context, dtype=torch.long), torch.tensor(target, dtype=torch.long))


In [8]:
class CBOW(nn.Module):
  def __init__(self, vocab_size, embedding_dim):
    super().__init__()
    self.emb = nn.Embedding(vocab_size, embedding_dim)
    self.lin = nn.Linear(embedding_dim, vocab_size)

  def forward(self, tens):
    out = self.emb(tens)

    srednia = torch.mean(out, dim=1)

    return self.lin(srednia)